In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

In [ ]:
RF  = pd.read_csv("../data/RF_Predictions_training_I.csv", index_col=0)
BRT = pd.read_csv("../data/BRT_Predictions_training_I.csv", index_col=0)
MLP = pd.read_csv("../data/MLP_Predictions_training_I.csv", index_col=0)
GAM = pd.read_csv("../data/GAM_Predictions_training_I.csv", index_col=0)

In [ ]:
RF  = RF.rename(columns={"Predicted": "RF_pred"})
BRT = BRT.rename(columns={"Predicted": "BRT_pred"})
MLP = MLP.rename(columns={"Predicted": "MLP_pred"})
GAM = GAM.rename(columns={"Predicted": "GAM_pred"})

In [ ]:
df_all = GAM[["Actual"]].copy()  

df_all["GAM_pred"] = GAM["GAM_pred"]
df_all["RF_pred"]  = RF["RF_pred"]
df_all["BRT_pred"] = BRT["BRT_pred"]
df_all["MLP_pred"] = MLP["MLP_pred"]

In [ ]:
model_cols = ["GAM_pred", "RF_pred", "BRT_pred", "MLP_pred"]

df_all["prediction_discrepancy"] = df_all[model_cols].std(axis=1)

In [ ]:
SHAPDM_ref_GAM = pd.read_csv("../data/GAM_ref_model_SHAP_discrepancy.csv")
SHAPDM_ref_RF = pd.read_csv("../data/RF_ref_model_SHAP_discrepancy.csv")
SHAPDM_ref_BRT = pd.read_csv("../data/BRT_ref_model_SHAP_discrepancy.csv")
SHAPDM_ref_MLP = pd.read_csv("../data/MLP_ref_model_SHAP_discrepancy.csv")

In [ ]:
df_all = df_all.reset_index(drop=True)

SHAPDM_ref_GAM["pred_dis"] = df_all["prediction_discrepancy"]
SHAPDM_ref_RF["pred_dis"]  = df_all["prediction_discrepancy"]
SHAPDM_ref_BRT["pred_dis"] = df_all["prediction_discrepancy"]
SHAPDM_ref_MLP["pred_dis"] = df_all["prediction_discrepancy"]

In [ ]:
for df in [SHAPDM_ref_GAM, SHAPDM_ref_RF, SHAPDM_ref_BRT, SHAPDM_ref_MLP]:
    df.rename(columns={"Mean_over_SDofMean": "shap_dis"}, inplace=True) #  inplace =True - this changes the exisiting dataframe instead of creating a new one.

In [ ]:
SHAP_all = pd.concat([
    SHAPDM_ref_GAM,
    SHAPDM_ref_RF,
    SHAPDM_ref_BRT,
    SHAPDM_ref_MLP
], ignore_index=True)

## Prediction discrepancy

### Realtive error of predictions

In [ ]:
df_all["relative_error"] = np.sqrt(
    (
        (df_all["GAM_pred"]  - df_all["Actual"])**2 +
        (df_all["RF_pred"]  - df_all["Actual"])**2 +
        (df_all["BRT_pred"] - df_all["Actual"])**2 +
        (df_all["MLP_pred"] - df_all["Actual"])**2 
    ) / 3
)

In [ ]:
SHAPDM_ref_GAM["relative_error"] = df_all["relative_error"]
SHAPDM_ref_RF["relative_error"] = df_all["relative_error"]
SHAPDM_ref_BRT["relative_error"] = df_all["relative_error"]
SHAPDM_ref_MLP["relative_error"] = df_all["relative_error"]

In [ ]:
SHAP_all = pd.concat([
    SHAPDM_ref_GAM.assign(ref="GAM"),
    SHAPDM_ref_RF.assign(ref="RF"),
    SHAPDM_ref_BRT.assign(ref="BRT"),
    SHAPDM_ref_MLP.assign(ref="MLP")
], ignore_index=True)


In [ ]:
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    "legend.fontsize": 12,
    "legend.title_fontsize": 14,
    "legend.frameon": False,
})

plt.figure(figsize=(7.5, 5))

palette = {
    "Observed": "black",   
    "GAM":"#C77CFF",     
    "RF": "#00A9FF",      
    "BRT": "#7CAE00",     
    "MLP": "#F8766D"
}

ax = sns.scatterplot(
    data=SHAP_all,
    x="relative_error",
    y="shap_dis",
    hue="ref",
    alpha=0.7,
    palette=palette,
)

plt.xlabel("Prediction discrepancy")
plt.ylabel("SHAP discrepancy")
# plt.title("Prediction discrepancy = prediction error from observed")

# Legend
order_legend=["RF","BRT","MLP","GAM"]
handles, labels = ax.get_legend_handles_labels()
label_to_handle = dict(zip(labels, handles))
ordered_handles = [label_to_handle[l] for l in order_legend if l in label_to_handle]
ordered_labels  = [l for l in order_legend if l in label_to_handle]

ax.legend(
    ordered_handles,
    ordered_labels,
    title="Reference model",  
    loc="center left",       
    bbox_to_anchor=(1.02, 0.5),  
    borderaxespad=0,
    frameon=False,
    ncol=1           
)


plt.tight_layout()
plt.show()